In [ ]:
!pip install -q transformers datasets sentencepiece accelerate jiwer tqdm

In [ ]:
import random, math, os
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import pandas as pd
import torch
from tqdm import tqdm
from jiwer import wer, cer
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)


In [ ]:
model_name = "t5-base"
num_synthetic_per_line = 3
target_dataset_size = 10000
max_length = 128
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
num_train_epochs = 8
learning_rate = 3e-4
output_dir = "./ocr_corrector_t5_base"

In [ ]:
print("Loading Teklia/IAM-line (this may take a little)...")
ds = load_dataset("Teklia/IAM-line", split="train")
lines = [ex["text"].strip() for ex in ds if ex["text"] and len(ex["text"].split()) > 1]

# keep only reasonably short lines (avoid very long lines)
lines = [l for l in lines if len(l) < 120]
print(f"Available clean lines: {len(lines)}")

# if not enough lines, fall back to wikitext
if len(lines) < 200:
    print("Too few lines found in Teklia/IAM-line; falling back to wikitext-2-raw-v1")
    ds2 = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    lines = [l["text"].strip() for l in ds2 if l["text"] and len(l["text"].split()) > 3]
    lines = [l for l in lines if len(l) < 120]
    print("Fallback lines:", len(lines))


In [ ]:
print("\nSample lines from the dataset:")
for l in lines[:10]:   # first 10 lines
    print("-", l)

In [ ]:
import random

# --- CONFUSION MAPS ---

# OCR-style (visual confusions)
ocr_confusions = {
    "m": ["rn"], "rn": ["m"],
    "o": ["0", "°"], "0": ["o"],
    "i": ["l", "1"], "l": ["i", "1"],
    "u": ["v"], "v": ["u"],
    "c": ["e"], "e": ["c"],
    "s": ["5", "$"], "5": ["s"],
    "g": ["q", "9"], "q": ["g"],
    "t": ["f"], "h": ["b"],
    "a": ["@", "ä"],
}

# Dyslexia-style (cognitive/phonetic confusions)
dys_confusions = {
    "b": ["d", "p"], "d": ["b", "q"], "p": ["q", "b"], "q": ["p", "d"],
    "t": ["d"], "f": ["ph"], "ph": ["f"],
    "k": ["c"], "c": ["k", "s"], "s": ["c", "z"], "z": ["s"],
    "m": ["n"], "n": ["m"], "r": ["l"], "l": ["r"],
    "v": ["w"], "w": ["v"], "g": ["j"], "j": ["g"],
    "y": ["i"], "i": ["y"],
}

# --- HYBRID NOISE FUNCTION ---

def inject_noise(
    text,
    mode="hybrid",  # "ocr", "dyslexic", or "hybrid"
    prob_confuse=0.15,
    prob_delete=0.05,
    prob_insert=0.05,
    prob_swap=0.07,
    prob_space=0.06,
    prob_case=0.05
):
    """Inject OCR-like, Dyslexic-like, or hybrid noise into text."""

    if mode == "ocr":
        confusion_map = ocr_confusions
    elif mode == "dyslexic":
        confusion_map = dys_confusions
    else:  # hybrid mode
        confusion_map = {**ocr_confusions, **dys_confusions}

    noisy = ""
    for ch in text:
        # random deletion
        if random.random() < prob_delete and ch != " ":
            continue

        # confusion substitution
        if ch.lower() in confusion_map and random.random() < prob_confuse:
            sub = random.choice(confusion_map[ch.lower()])
            noisy += sub.upper() if ch.isupper() else sub
        else:
            noisy += ch

        # random insertion (like dyslexic doubling)
        if random.random() < prob_insert:
            noisy += ch

    # transpositions
    if random.random() < prob_swap and len(noisy) > 3:
        i = random.randint(0, len(noisy) - 2)
        noisy = noisy[:i] + noisy[i+1] + noisy[i] + noisy[i+2:]

    # random spacing errors
    if random.random() < prob_space:
        idx = random.randint(0, len(noisy) - 1)
        noisy = noisy[:idx] + " " + noisy[idx:]
    if random.random() < prob_space and " " in noisy:
        noisy = noisy.replace(" ", "", 1)

    # case irregularity
    if random.random() < prob_case and len(noisy) > 2:
        idx = random.randint(0, len(noisy) - 1)
        noisy = noisy[:idx] + noisy[idx].swapcase() + noisy[idx+1:]

    return noisy

# --- DEMO TEST ---
samples = [
    "This is a test sentence",
    "Machine learning for OCR is fun",
    "Dyslexic handwriting is hard to read",
    "The quick brown fox jumps over the lazy dog",
    "Please write your name on the paper"
]

for s in samples:
    print("Clean:", s)
    print("OCR Noise :", inject_noise(s, mode="ocr"))
    print("Dyslexic :", inject_noise(s, mode="dyslexic"))
    print("Hybrid :", inject_noise(s, mode="hybrid"))
    print("-" * 60)


In [ ]:
lines = samples
target_dataset_size = 2000

print("\nBuilding synthetic dataset...")
pairs = []
pbar = tqdm(total=target_dataset_size)
while len(pairs) < target_dataset_size:
    line = random.choice(lines)
    noisy = inject_noise(line, mode="hybrid")
    if len(noisy.strip()) == 0 or noisy.strip() == line.strip():
        continue
    pairs.append({"noisy": noisy, "clean": line})
    pbar.update(1)
pbar.close()
print("Total pairs:", len(pairs))

df = pd.DataFrame(pairs)
dataset = Dataset.from_pandas(df)
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]

print("Train size:", len(train_ds), "Eval size:", len(eval_ds))


In [ ]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

max_length = 64

def preprocess(batch):
    inputs = tokenizer(batch["noisy"], padding="max_length", truncation=True, max_length=max_length)
    labels = tokenizer(batch["clean"], padding="max_length", truncation=True, max_length=max_length)
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=["noisy","clean"])
eval_tokenized = eval_ds.map(preprocess, batched=True, remove_columns=["noisy","clean"])
print("Tokenized columns:", train_tokenized.column_names)


In [ ]:
output_dir = "./text_correction_hybrid"
learning_rate = 3e-4
per_device_train_batch_size = 8
gradient_accumulation_steps = 2
num_train_epochs = 3

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    logging_dir="./logs",
    learning_rate=learning_rate,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    save_total_limit=3,
    fp16=True if device=="cuda" else False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training finished. Saving model...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)



In [ ]:
print("\nRunning inference on eval set and computing WER/CER...")

def model_correct(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    out = model.generate(**enc, max_length=max_length, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample_eval = eval_ds.select(range(min(50, len(eval_ds))))
rows, wer_list, cer_list = [], [], []

for ex in tqdm(sample_eval):
    noisy, clean = ex["noisy"], ex["clean"]
    pred = model_correct(noisy)
    rows.append({"noisy": noisy, "clean": clean, "pred": pred})
    try:
        wer_list.append(wer(clean, pred))
        cer_list.append(cer(clean, pred))
    except Exception:
        pass

results_df = pd.DataFrame(rows)
print(f"\nAvg WER (sample): {sum(wer_list)/len(wer_list):.3f}  Avg CER (sample): {sum(cer_list)/len(cer_list):.3f}")
print(results_df.head(10))

results_df.to_csv("correction_results_sample.csv", index=False)
print("Saved correction_results_sample.csv and model at", output_dir)

In [ ]:
from google.colab import files
import shutil

# Step 1: Zip your model folder
shutil.make_archive('text_correction_hybrid', 'zip', './text_correction_hybrid')

# Step 2: Download the zip file
files.download('text_correction_hybrid.zip')
